# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve all record sets in the dataset by their `@id`
record_sets = dataset.metadata.record_sets()
print("Record Sets with their '@id':")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")
    # List fields for this record set
    if 'field' in rs:
        print("    Fields and their '@id':")
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) else field
            field_obj = dataset.metadata.field_by_id(field_id)
            print(f"      - {field_id}: {field_obj.get('name', '(no name)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Convenience helper functions for Croissant schema navigation
def get_record_set_ids(dataset):
    return [rs['@id'] for rs in dataset.metadata.record_sets()]

# List all record set @ids
record_set_ids = get_record_set_ids(dataset)
print("Record set '@id's found:")
for rid in record_set_ids:
    print(rid)

# Extract data from all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from '{record_set_id}'...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Shape: {df.shape}")
    except Exception as e:
        print(f"  Failed to load records: {e}")

# Pick the first record set for demonstration (replace with your `@id` as needed)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA on the first available record set
import numpy as np

# Inspect column types in the main record set
df = dataframes[main_rs_id]
# Select a likely numeric field for demo: try common names
numeric_candidates = ['MW_kDa', 'MW', 'Abundance', 'normalized_abundance', 'peptide_count', 'coverage_percentage']
numeric_field = None
for col in df.columns:
    if col.lower() in [n.lower() for n in numeric_candidates]:
        numeric_field = col
        break

if numeric_field is None:
    # Fallback: pick first column with numeric dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field is not None:
    print(f"Using numeric field for analysis: {numeric_field}")

    # Drop NA and ensure numeric
    df = df.copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.75)  # Use a high quantile as demonstration
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_candidates = ['sample', 'condition', 'accession', 'label']
    group_field = None
    for col in df.columns:
        if col.lower() in [g.lower() for g in group_candidates]:
            group_field = col
            break

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, we'll plot the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group field available, boxplot per group
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df.dropna(subset=[numeric_field]), x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:  
- Load and inspect Croissant-based mass spectrometry datasets using `mlcroissant`  
- Explore the schema: record sets, fields, and their `@id` identifiers  
- Extract records to DataFrames by referencing their `@id`  
- Run standard EDA operations and visualize data distributions  

This workflow can be adapted for similar FAIR datasets described in the Croissant format.